## ASSUMPTIONS ##

This script assumes that none of the rules like consistency, daily drawdown etc. got/get breached.

In [70]:
import plotly.graph_objects as go
import numpy as np
import math

# Variables

# General

account_size = 25000
account_price = 70
return_per_trade = 0.01
fee_per_trade_live_account = 2  # IBKR + Puffer

# Eval

profit_target = 0.05
max_drawdown = 0.04

# Funded

funded_payout_target = 0.06
funded_max_drawdown = 0.04

P_FAIL_EVAL, P_PAYOUT, P_FAIL_FUNDED = 0.5, 0.25, 0.25


def n_trades_with_fees(V0, target, r, fee):
    assert V0 * r > fee, "Fee zu hoch relativ zu Return/Kapital - Ziel nie erreichbar"
    x = (V0 * (1 + target) - fee / r) / (V0 - fee / r)
    return math.log(x) / math.log(1 + r)


funded_start_capital = account_size * (1 + profit_target)

fee_per_trade_eval_account = 5  # ggf. anpassen, falls Eval != Live-Kontraktgröße

n_trades_eval = n_trades_with_fees(
    account_size, profit_target, return_per_trade, fee_per_trade_eval_account
)
n_trades_funded = n_trades_with_fees(
    funded_start_capital, funded_payout_target, return_per_trade, fee_per_trade_live_account
)


def compute_metrics(evaluation_hedge, funded_hedge):
    fees_eval = n_trades_eval * fee_per_trade_eval_account
    pass_eval_pnl = -int(profit_target * account_size * evaluation_hedge) - account_price - fees_eval
    fail_eval_pnl = -account_price + max_drawdown * account_size * evaluation_hedge - fees_eval

    fees_funded = n_trades_funded * fee_per_trade_live_account
    payout = funded_start_capital * funded_payout_target
    funded_hedge_cost = funded_hedge * funded_start_capital
    fail_funded_pnl = pass_eval_pnl + (funded_hedge_cost * funded_max_drawdown) - fees_funded
    payout_pnl = pass_eval_pnl - (funded_hedge_cost * funded_payout_target) + payout - fees_funded

    outcomes = [fail_eval_pnl, payout_pnl, fail_funded_pnl]
    probs = [P_FAIL_EVAL, P_PAYOUT, P_FAIL_FUNDED]

    ev = sum(p * x for p, x in zip(probs, outcomes))
    variance = sum(p * (x - ev) ** 2 for p, x in zip(probs, outcomes))
    downside_var = sum(p * min(x - ev, 0) ** 2 for p, x in zip(probs, outcomes))
    downside_std = math.sqrt(downside_var)

    invested = account_price + funded_hedge_cost
    sortino = ev / downside_std if downside_std > 0 else float("nan")

    return {
        "ev": ev, "std": math.sqrt(variance), "sortino": sortino, "roi": ev / invested,
        "pass_eval_pnl": pass_eval_pnl, "fail_eval_pnl": fail_eval_pnl,
        "payout_pnl": payout_pnl, "fail_funded_pnl": fail_funded_pnl,
    }


def breakeven_probabilities(evaluation_hedge, funded_hedge, p_fail_eval=P_FAIL_EVAL):
    
    m = compute_metrics(evaluation_hedge, funded_hedge)
    fail_eval_pnl = m["fail_eval_pnl"]
    payout_pnl = m["payout_pnl"]
    fail_funded_pnl = m["fail_funded_pnl"]

    denom_payout = payout_pnl - fail_funded_pnl
    degenerate = abs(denom_payout) < 1e-6

    p_payout_breakeven = (
        float("nan") if degenerate else
        -(p_fail_eval * fail_eval_pnl + (1 - p_fail_eval) * fail_funded_pnl) / denom_payout
    )

    avg_funded_pnl = 0.5 * payout_pnl + 0.5 * fail_funded_pnl
    denom_pass = avg_funded_pnl - fail_eval_pnl
    p_pass_eval_breakeven = float("nan") if denom_pass == 0 else -fail_eval_pnl / denom_pass
    # Ausserhalb [0,1] gibt es rechnerisch keine echte Breakeven-Wahrscheinlichkeit -
    # z.B. wenn alle drei Outcomes das gleiche Vorzeichen haben (immer profitabel/
    # immer verlustreich, unabhaengig von der Win-Rate).
    pass_eval_valid = 0.0 <= p_pass_eval_breakeven <= 1.0

    return {
        "p_payout_breakeven": p_payout_breakeven,
        "degenerate": degenerate,
        "p_pass_eval_breakeven": p_pass_eval_breakeven,
        "pass_eval_valid": pass_eval_valid,
        "fail_eval_pnl": fail_eval_pnl,
        "payout_pnl": payout_pnl,
        "fail_funded_pnl": fail_funded_pnl,
    }


def capital_requirement(evaluation_hedge, funded_hedge,
                         index_price=7495, mes_margin_per_contract=50,
                         margin_safety_buffer=0.25):

    m = compute_metrics(evaluation_hedge, funded_hedge)

    worst_running_pnl = min(
        0, m["pass_eval_pnl"], m["fail_eval_pnl"], m["payout_pnl"], m["fail_funded_pnl"]
    )

    mes_contract_value = 5 * index_price
    eval_hedge_notional = evaluation_hedge * account_size
    funded_hedge_notional = funded_hedge * funded_start_capital

    n_contracts_eval = math.ceil(eval_hedge_notional / mes_contract_value)
    n_contracts_funded = math.ceil(funded_hedge_notional / mes_contract_value)
    max_contracts = max(n_contracts_eval, n_contracts_funded)

    margin_required = max_contracts * mes_margin_per_contract * (1 + margin_safety_buffer)
    total_capital = abs(worst_running_pnl) + margin_required

    return {
        "worst_running_pnl": worst_running_pnl,
        "n_contracts_eval": n_contracts_eval,
        "n_contracts_funded": n_contracts_funded,
        "max_contracts": max_contracts,
        "margin_required": margin_required,
        "total_capital": total_capital,
    }


def optimize_hedges(eval_range=np.arange(0.0, 1.01, 0.05),
                     funded_range=np.arange(0.0, 1.01, 0.05),
                     metric="sortino"):
    z = np.zeros((len(eval_range), len(funded_range)))
    best = {"value": -math.inf}

    for i, eh in enumerate(eval_range):
        for j, fh in enumerate(funded_range):
            m = compute_metrics(eh, fh)
            z[i, j] = m[metric]
            if not math.isnan(m[metric]) and m[metric] > best["value"]:
                best = {"value": m[metric], "evaluation_hedge": eh, "funded_hedge": fh, "metrics": m}

    fig = go.Figure(data=go.Heatmap(z=z, x=funded_range, y=eval_range, colorscale="Viridis",
                                     colorbar=dict(title=metric)))
    fig.add_trace(go.Scatter(x=[best["funded_hedge"]], y=[best["evaluation_hedge"]],
                              mode="markers+text", marker=dict(color="red", size=14, symbol="x"),
                              text=["Optimum"], textposition="top center", showlegend=False))
    fig.update_layout(
        title=f"{metric} nach Hedge-Ratio (Optimum: eval={best['evaluation_hedge']:.2f}, funded={best['funded_hedge']:.2f})",
        xaxis_title="Funded Hedge", yaxis_title="Evaluation Hedge",
        template="plotly_dark", width=900, height=650,
    )
    fig.show()

    return best


def plot_tree(evaluation_hedge, funded_hedge):
    m = compute_metrics(evaluation_hedge, funded_hedge)
    fig = go.Figure()

    def add_node(x, y, title, subtitle):
        fig.add_annotation(x=x, y=y, text=f"<b>{title}</b><br><span style='font-size:12px'>{subtitle}</span>",
                            showarrow=False, bgcolor="#19321d", borderwidth=0, borderpad=10,
                            font=dict(color="white", size=15), align="center")

    def add_line(x0, y0, x1, y1):
        fig.add_shape(type="line", x0=x0, x1=x1, y0=y0, y1=y1, line=dict(color="white", width=1))

    add_node(3, 6, "Start Challenge", f"Fee ${-account_price}, hedge {int(evaluation_hedge*100)}%")
    add_line(3, 6, 3, 4)
    add_line(3, 6, 1, 4)

    add_node(3, 4, "Pass Evaluation", f"Running ${int(m['pass_eval_pnl'])}, hedge {int(funded_hedge*100)}%")
    add_node(1, 4, "Fail Evaluation", f"Net ${int(m['fail_eval_pnl'])}")

    add_node(3, 2, "Funded Payout", f"Running ${int(m['payout_pnl'])}")
    add_node(1, 2, "Fail Funded", f"Net ${int(m['fail_funded_pnl'])}")
    add_line(3, 4, 3, 2)
    add_line(3, 4, 1, 2)

    fig.update_layout(
        title=f"Optimale Ratios: eval={evaluation_hedge:.2f}, funded={funded_hedge:.2f} | "
              f"EV=${int(m['ev'])}, Sortino={m['sortino']:.2f}",
        template="plotly_dark",
        xaxis=dict(visible=False, range=[0, 6]),
        yaxis=dict(visible=False, range=[0, 6]),
        showlegend=False, margin=dict(l=20, r=20, t=60, b=20),
        width=1000, height=600,
    )
    fig.show()


if __name__ == "__main__":
    best = optimize_hedges(metric="sortino")
    eh, fh = best["evaluation_hedge"], best["funded_hedge"]
    plot_tree(eh, fh)

    be = breakeven_probabilities(eh, fh)
    print(f"\nOptimales Hedge-Ratio: eval={eh:.2f}, funded={fh:.2f}")
    if be["degenerate"]:
        print("payout_pnl ~= fail_funded_pnl -> Payout-Wahrscheinlichkeit hat hier "
              "keinen Einfluss mehr auf EV (Hedge macht Funded-Stufe risikoneutral).")
    else:
        print(f"Breakeven Payout-Wahrscheinlichkeit (unconditional): {be['p_payout_breakeven']:.2%} "
              f"(aktuell angenommen: {P_PAYOUT:.2%})")

    if be["pass_eval_valid"]:
        print(f"Breakeven Eval-Pass-Rate: {be['p_pass_eval_breakeven']:.2%} "
              f"(aktuell angenommen: {1 - P_FAIL_EVAL:.2%})")
    else:
        sign_note = "positiv" if be["fail_eval_pnl"] > 0 else "negativ"
        print(f"Keine gueltige Breakeven-Wahrscheinlichkeit in [0,1]: fail_eval_pnl (${be['fail_eval_pnl']:.0f}), "
              f"payout_pnl (${be['payout_pnl']:.0f}) und fail_funded_pnl (${be['fail_funded_pnl']:.0f}) "
              f"sind alle {sign_note} -> EV ist bei diesem Hedge-Ratio unabhaengig von der Win-Rate {sign_note}.")

    cap = capital_requirement(eh, fh)
    print(f"\nSchlechtester Running PnL entlang des Pfades: ${cap['worst_running_pnl']:.0f}")
    print(f"MES-Kontrakte Eval: {cap['n_contracts_eval']}, Funded: {cap['n_contracts_funded']}")
    print(f"Benoetigte Margin (inkl. Puffer): ${cap['margin_required']:.0f}")
    print(f"Gesamt benoetigtes Kapital: ${cap['total_capital']:.0f}")


Optimales Hedge-Ratio: eval=0.25, funded=0.60
payout_pnl ~= fail_funded_pnl -> Payout-Wahrscheinlichkeit hat hier keinen Einfluss mehr auf EV (Hedge macht Funded-Stufe risikoneutral).
Keine gueltige Breakeven-Wahrscheinlichkeit in [0,1]: fail_eval_pnl ($155), payout_pnl ($211) und fail_funded_pnl ($211) sind alle positiv -> EV ist bei diesem Hedge-Ratio unabhaengig von der Win-Rate positiv.

Schlechtester Running PnL entlang des Pfades: $-407
MES-Kontrakte Eval: 1, Funded: 1
Benoetigte Margin (inkl. Puffer): $62
Gesamt benoetigtes Kapital: $470
